# 08 — Surface code intro: lattice, stabilizers, syndromes

Phase 8 demo. We build `d = 3` and `d = 5` rotated planar surface codes, visualise the lattice, print the stabilizer generators, inject errors and read off the resulting syndrome detection events.

We **do not** build a decoder here; the standard MWPM decoder (PyMatching) is a follow-on, listed as out-of-scope in the project plan.

Read alongside `notes/11-surface-code.md`.

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection

from qec.pauli import Pauli
from qec.codes.surface import SurfaceCode, syndrome


## 1. d=3 surface code: 9 data qubits, 8 stabilizers

`d^2 = 9` data qubits arranged on a 3×3 grid; `d^2 - 1 = 8` stabilizer generators (4 X + 4 Z), giving 1 logical qubit.


In [ ]:
code = SurfaceCode(d=3)
xs, zs = code.stabilizer_generators()
print(f"data qubits : {code.n}")
print(f"X-stabilizers ({len(xs)}):")
for s in xs: print(" ", s, " weight", s.weight())
print(f"Z-stabilizers ({len(zs)}):")
for s in zs: print(" ", s, " weight", s.weight())
print(f"logical X (weight {code.logical_x().weight()}): {code.logical_x()}")
print(f"logical Z (weight {code.logical_z().weight()}): {code.logical_z()}")


## 2. Visualise the lattice

Each data qubit is a circle; X-stabilizers are red faces, Z-stabilizers blue.


In [ ]:
def lattice_plot(code, x_violations=None, z_violations=None, error_qubits=None,
                  title=None, ax=None):
    """Draw a d x d surface-code lattice. Highlight stabilizer violations and
    error qubits if provided."""
    d = code.d
    if ax is None:
        fig, ax = plt.subplots(figsize=(4 + d * 0.5, 4 + d * 0.5))
    xs, zs = code.stabilizer_generators()

    # Plaquette polygons. We approximate each plaquette by the convex hull of
    # the qubits it acts on.
    def patch_for(stab):
        qs = [q for q in range(code.n) if stab.x[q] or stab.z[q]]
        pts = [(q % d, -(q // d)) for q in qs]  # x = col, y = -row
        # Use bounding box for visual clarity (works for both bulk + boundary).
        xs_p = [p[0] for p in pts]; ys_p = [p[1] for p in pts]
        return Polygon(
            [(min(xs_p) - 0.4, min(ys_p) - 0.4), (max(xs_p) + 0.4, min(ys_p) - 0.4),
             (max(xs_p) + 0.4, max(ys_p) + 0.4), (min(xs_p) - 0.4, max(ys_p) + 0.4)],
            closed=True,
        )

    x_patches = [patch_for(s) for s in xs]
    z_patches = [patch_for(s) for s in zs]

    # X-stabilizers: red, with stronger fill if violated.
    for i, p in enumerate(x_patches):
        violated = x_violations is not None and x_violations[i]
        ax.add_patch(Polygon(p.get_xy(), closed=True,
                              facecolor="red", alpha=0.35 if violated else 0.10,
                              edgecolor="darkred"))
    for i, p in enumerate(z_patches):
        violated = z_violations is not None and z_violations[i]
        ax.add_patch(Polygon(p.get_xy(), closed=True,
                              facecolor="blue", alpha=0.35 if violated else 0.10,
                              edgecolor="darkblue"))

    # Data qubits as circles
    for r in range(d):
        for c in range(d):
            q = code.qubit(r, c)
            color = "black"
            if error_qubits is not None and q in error_qubits:
                color = "orange"
            ax.plot(c, -r, "o", markersize=14, markerfacecolor="white",
                    markeredgecolor=color, markeredgewidth=2)
            ax.annotate(str(q), (c, -r), ha="center", va="center", fontsize=8)

    ax.set_xlim(-0.7, d - 0.3)
    ax.set_ylim(-d + 0.3, 0.7)
    ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title)
    return ax

fig, ax = plt.subplots(figsize=(5, 5))
lattice_plot(code, title="d = 3 surface code (clean)", ax=ax)
plt.show()


## 3. Single Z error: two X-stabilizers light up

A Z error on a bulk qubit flips the two X-stabilizers it touches — a pair of detection events that the (would-be) decoder pairs up to a single Z error.


In [ ]:
# Inject Z on qubit 4 (centre)
err_x = np.zeros(code.n, dtype=np.int8); err_z = np.zeros(code.n, dtype=np.int8)
err_z[4] = 1
E = Pauli(err_x, err_z, 0)
xv, zv = syndrome(code, E)
print("X-stab violations:", xv, "  (positions of detection events)")
print("Z-stab violations:", zv)
fig, ax = plt.subplots(figsize=(5, 5))
lattice_plot(code, x_violations=xv, z_violations=zv, error_qubits={4},
             title="single Z error on qubit 4", ax=ax)
plt.show()


## 4. A horizontal X-string on the top row

In our auto-derived logicals, logical X is `X` on every qubit of the top row. Apply it: this is a logical operator, so it must produce *no* detection events (the "string" goes from one X-boundary to the other).


In [ ]:
LX = code.logical_x()
print("logical X:", LX)
xv, zv = syndrome(code, LX)
print("syndrome of logical X:", xv, zv,
      " (must all be zero for a logical operator)")
assert xv == [0]*len(xv) and zv == [0]*len(zv)
fig, ax = plt.subplots(figsize=(5, 5))
err_qs = {q for q in range(code.n) if LX.x[q] or LX.z[q]}
lattice_plot(code, x_violations=xv, z_violations=zv, error_qubits=err_qs,
             title="logical X chain (no detection events)", ax=ax)
plt.show()


## 5. d = 5: 25 data qubits, 24 stabilizers

The same picture at distance 5 — illustrating the polynomial scaling that the surface code buys you.


In [ ]:
code5 = SurfaceCode(d=5)
xs5, zs5 = code5.stabilizer_generators()
print(f"d = 5: {code5.n} data, {len(xs5) + len(zs5)} stabilizers")
print(f"logical X weight: {code5.logical_x().weight()}  (= d)")
print(f"logical Z weight: {code5.logical_z().weight()}  (= d)")

fig, ax = plt.subplots(figsize=(7, 7))
# Inject a Z error pattern that "almost" forms a logical Z chain (length d - 1)
err_x = np.zeros(code5.n, dtype=np.int8); err_z = np.zeros(code5.n, dtype=np.int8)
for r in range(4):  # length-4 vertical chain on column 0 (one short of logical)
    err_z[code5.qubit(r, 0)] = 1
E = Pauli(err_x, err_z, 0)
xv, zv = syndrome(code5, E)
print(f"length-4 Z chain on column 0: X-violations = {sum(xv)}, Z-violations = {sum(zv)}")

err_qs = {q for q in range(code5.n) if E.x[q] or E.z[q]}
lattice_plot(code5, x_violations=xv, z_violations=zv, error_qubits=err_qs,
             title="d = 5: length-4 Z chain (sub-distance, recoverable)", ax=ax)
plt.show()


## What this notebook gates

- The d = 3 surface code has 9 data qubits, 8 stabilizers (4 X + 4 Z), 1 logical qubit, distance 3.
- Single Z errors produce paired X-stabilizer detection events whose endpoints are the error location.
- Logical operators are weight-`d` strings spanning the lattice; they produce zero syndrome (by definition).
- The pattern scales to d = 5 with 25 data qubits, 24 stabilizers, and weight-5 logicals.

This finishes Phase 8. The follow-on work — running circuit-level noise with Stim, MWPM decoding with PyMatching, and producing a real threshold plot — is parked as out-of-scope per the project plan.

Next: Phase 9 — capstone summary (`notes/12-summary.md`) and grand-tour notebook (`demos/00_grand_tour.ipynb`).
